In [15]:
import pandas as pd
import numpy as np
import json
import joblib
from math import sqrt

from sklearn.metrics import mean_absolute_error, mean_squared_error
from tensorflow.keras.models import load_model


In [16]:
with open("../saved_models/features.json") as f:
    feature_names = json.load(f)

print("[INFO] Loaded feature list:", len(feature_names))


[INFO] Loaded feature list: 17


In [17]:
EXTERNAL_PATH = "../data/external/external_dataset2.csv"

df_ext_raw = pd.read_csv(EXTERNAL_PATH)
print("[INFO] External raw data shape:", df_ext_raw.shape)

df_ext_raw.head()


[INFO] External raw data shape: (43825, 20)


,time,temperature,rain,snowfall,apparent_temperature,dew_point,humidity,precipitation,snow_depth,wind_speed_10m,wind_speed_100m,wind_direction_10m,wind_direction_100m,wind_gusts_10m,load_actual,load_forecast,solar_generation,wind_generation,BE_wind_offshore_generation_actual,BE_wind_onshore_generation_actual
0,2014-12-31T23:00,0.5,0.0,0.0,-3.2,-0.1,96,0.0,0.03,10.6,23.0,215,230,18.0,NaN,NaN,NaN,NaN,NaN,NaN
1,2015-01-01T00:00,0.1,0.0,0.0,-3.7,-0.5,95,0.0,0.03,10.9,23.7,214,227,18.4,9484.0,9897.0,NaN,NaN,NaN,NaN
2,2015-01-01T01:00,-0.2,0.0,0.0,-4.3,-0.9,95,0.0,0.03,12.4,25.0,210,220,19.1,9152.0,9521.0,NaN,734.81,518.66,216.15
3,2015-01-01T02:00,-0.4,0.0,0.0,-4.7,-1.2,95,0.0,0.03,13.5,26.3,209,216,20.2,8799.0,9135.0,NaN,766.64,529.46,237.18
4,2015-01-01T03:00,-0.6,0.0,0.0,-5.1,-1.3,95,0.0,0.03,14.9,28.2,200,208,22.0,8567.0,8909.0,NaN,733.13,406.94,326.19


In [18]:
import sys
import os

project_root = os.path.abspath("..")
sys.path.append(project_root)

from src.preprocessing.load_data import load_dataset
from src.preprocessing.clean_data import clean_data
from src.preprocessing.feature_engineering import add_features

# Load & standardize columns
df_ext = load_dataset(EXTERNAL_PATH)

# Clean data
df_ext = clean_data(df_ext)

# Feature engineering
df_ext = add_features(df_ext)

print("[INFO] External dataset after preprocessing:", df_ext.shape)
df_ext.head()


[INFO] Loading dataset from: ../data/external/external_dataset2.csv
[INFO] Final standardized columns: ['timestamp', 'load_actual', 'temperature', 'temperature', 'humidity', 'dew_point', 'solar_generation', 'wind_generation']
[INFO] Rows loaded: 43825
[INFO] Cleaning dataset...
[INFO] Data cleaning complete. Rows: 43825
[INFO] Adding calendar, lag, and rolling features...
[INFO] Feature engineering complete.
[INFO] External dataset after preprocessing: (43657, 19)


e:\Projects\LoadForecasting\aimlloadforecasting_b4\src\preprocessing\clean_data.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")


,timestamp,load_actual,temperature,humidity,dew_point,solar_generation,wind_generation,hour,day_of_week,month,is_weekend,hour_sin,hour_cos,load_lag_1,load_lag_24,load_lag_168,rolling_mean_24,rolling_std_24,rolling_mean_168
0,2015-01-07 23:00:00,10996.0,3.1,89,1.5,0.0,1422.66,23,2,1,0,-0.258819,0.965926,11735.0,11079.0,9484.0,11939.958333,1073.912776,10687.821429
1,2015-01-08 00:00:00,10519.0,3.6,85,1.2,0.0,1411.69,0,3,1,0,0.000000,1.000000,10996.0,10511.0,9484.0,11940.291667,1073.451098,10693.982143
2,2015-01-08 01:00:00,10083.0,4.2,82,1.4,0.0,1459.74,1,3,1,0,0.258819,0.965926,10519.0,10177.0,9152.0,11936.375000,1080.314037,10699.523810
3,2015-01-08 02:00:00,9803.0,4.7,83,2.0,0.0,1456.87,2,3,1,0,0.500000,0.866025,10083.0,10019.0,8799.0,11927.375000,1097.741191,10705.500000
4,2015-01-08 03:00:00,9828.0,5.0,86,2.8,0.0,1410.53,3,3,1,0,0.707107,0.707107,9803.0,10003.0,8567.0,11920.083333,1111.573539,10713.005952


In [19]:
TARGET = "load_actual"

X_ext = df_ext[feature_names]
y_ext = df_ext[TARGET].values

print("[INFO] External features shape:", X_ext.shape)
print("[INFO] External target shape:", y_ext.shape)


[INFO] External features shape: (43657, 17)
[INFO] External target shape: (43657,)


In [20]:
from math import sqrt
from sklearn.metrics import mean_absolute_error, mean_squared_error

def mape(y_true, y_pred):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)

    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100


In [21]:
rf = joblib.load("../saved_models/rf_model.pkl")
pred_rf = rf.predict(X_ext)

print("Random Forest (External)")
print("MAE :", mean_absolute_error(y_ext, pred_rf))
print("RMSE:", sqrt(mean_squared_error(y_ext, pred_rf)))
print("MAPE:", mape(y_ext, pred_rf))


Random Forest (External)
MAE : 46.270367982225075
RMSE: 72.99882028824902
MAPE: 0.4743205352630202


In [22]:
xgb = joblib.load("../saved_models/xgb_model.pkl")
pred_xgb = xgb.predict(X_ext)

print("XGBoost (External)")
print("MAE :", mean_absolute_error(y_ext, pred_xgb))
print("RMSE:", sqrt(mean_squared_error(y_ext, pred_xgb)))
print("MAPE:", mape(y_ext, pred_xgb))


XGBoost (External)
MAE : 57.442793171412944
RMSE: 78.30347182796562
MAPE: 0.5884886292367197


In [23]:
svm = joblib.load("../saved_models/svm_model.pkl")
svm_scaler = joblib.load("../saved_models/svm_scaler.pkl")

X_ext_scaled = svm_scaler.transform(X_ext)
pred_svm = svm.predict(X_ext_scaled)

print("SVM (External)")
print("MAE :", mean_absolute_error(y_ext, pred_svm))
print("RMSE:", sqrt(mean_squared_error(y_ext, pred_svm)))
print("MAPE:", mape(y_ext, pred_svm))


SVM (External)
MAE : 108.13714519375606
RMSE: 147.94054915169363
MAPE: 1.0991555855902515


In [24]:
arima = joblib.load("../saved_models/arima_model.pkl")
pred_arima = arima.forecast(steps=len(y_ext))

print("ARIMA (External)")
print("MAE :", mean_absolute_error(y_ext, pred_arima))
print("RMSE:", sqrt(mean_squared_error(y_ext, pred_arima)))
print("MAPE:", mape(y_ext, pred_arima))


ARIMA (External)
MAE : 1158.7090834350927
RMSE: 1384.7946797215927
MAPE: 12.282785463659094


In [25]:
def create_sequences(X, lookback=24):
    seq = []
    for i in range(lookback, len(X)):
        seq.append(X[i-lookback:i])
    return np.array(seq)

lookback = 24


In [26]:
scaler_X = joblib.load("../saved_models/cnn_lstm_scaler_X.pkl")
scaler_y = joblib.load("../saved_models/cnn_lstm_scaler_y.pkl")

X_ext_scaled = scaler_X.transform(X_ext)
X_ext_seq = create_sequences(X_ext_scaled, lookback)

y_ext_seq = y_ext[lookback:]

cnn_lstm = load_model("../saved_models/cnn_lstm_model.h5", compile=False)
pred_scaled = cnn_lstm.predict(X_ext_seq)
pred = scaler_y.inverse_transform(pred_scaled)

print("CNN-LSTM (External)")
print("MAE :", mean_absolute_error(y_ext_seq, pred))
print("RMSE:", sqrt(mean_squared_error(y_ext_seq, pred)))
print("MAPE:", mape(y_ext_seq, pred))


e:\Projects\LoadForecasting\aimlloadforecasting_b4\lfvenv\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but MinMaxScaler was fitted without feature names
  warnings.warn(


1364/1364 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step
CNN-LSTM (External)
MAE : 149.1086818292992
RMSE: 201.98635347708634
MAPE: 1.5264843161458506


In [27]:
results = [
    ["Random Forest", *[
        mean_absolute_error(y_ext, pred_rf),
        sqrt(mean_squared_error(y_ext, pred_rf)),
        mape(y_ext, pred_rf)
    ]],
    ["XGBoost", *[
        mean_absolute_error(y_ext, pred_xgb),
        sqrt(mean_squared_error(y_ext, pred_xgb)),
        mape(y_ext, pred_xgb)
    ]],
    ["SVM", *[
        mean_absolute_error(y_ext, pred_svm),
        sqrt(mean_squared_error(y_ext, pred_svm)),
        mape(y_ext, pred_svm)
    ]],
    ["ARIMA", *[
        mean_absolute_error(y_ext, pred_arima),
        sqrt(mean_squared_error(y_ext, pred_arima)),
        mape(y_ext, pred_arima)
    ]],
    ["CNN-LSTM", *[
        mean_absolute_error(y_ext_seq, pred),
        sqrt(mean_squared_error(y_ext_seq, pred)),
        mape(y_ext_seq, pred)
    ]]
]

external_results_df = pd.DataFrame(
    results,
    columns=["Model", "MAE", "RMSE", "MAPE (%)"]
).sort_values("MAPE (%)")

external_results_df


,Model,MAE,RMSE,MAPE (%)
0,Random Forest,46.270368,72.998820,0.474321
1,XGBoost,57.442793,78.303472,0.588489
2,SVM,108.137145,147.940549,1.099156
4,CNN-LSTM,149.108682,201.986353,1.526484
3,ARIMA,1158.709083,1384.794680,12.282785


In [29]:
external_results_df.to_csv("../saved_models/external_dataset_results.csv", index=False)
print("[INFO] External dataset evaluation saved.")


[INFO] External dataset evaluation saved.
